# Azure OpenAI connection test

Run these cells from the project root environment. They check `.env` loading, make a direct Azure OpenAI call, and optionally call the local FastAPI agent endpoint.

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / ".env").exists():
    # If the notebook kernel starts inside notebooks/, step back to the repo root.
    candidate = project_root.parent
    if (candidate / ".env").exists():
        project_root = candidate

env_path = project_root / ".env"
loaded = load_dotenv(env_path, override=True)

required = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
]

def mask(value: str | None) -> str:
    if not value:
        return "<missing>"
    if len(value) <= 10:
        return "<set>"
    return f"{value[:6]}...{value[-4:]}"

print(f"Project root: {project_root}")
print(f".env path: {env_path}")
print(f".env loaded: {loaded}")
print()

missing = []
for key in required:
    value = os.getenv(key)
    print(f"{key}: {mask(value)}")
    if not value:
        missing.append(key)

if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]

response = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": "Reply with exactly one word: pong"}],
    max_completion_tokens=16,
)

print("Direct Azure OpenAI call succeeded.")
print("Deployment:", deployment)
print("Response:", response.choices[0].message.content)

In [ ]:
import requests

api_base_url = os.getenv("API_BASE_URL", "http://localhost:8000")

health = requests.get(f"{api_base_url}/", timeout=10)
print("FastAPI health status:", health.status_code)
print(health.json())

agent_response = requests.post(
    f"{api_base_url}/agent/process",
    json={
        "prompt": "Say hello and do not call any tools.",
        "model_name": os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    },
    timeout=120,
)

print("Agent endpoint status:", agent_response.status_code)
print(agent_response.json())